In [1]:
import math
import re
import numpy as np


np.set_printoptions(precision=4, suppress=True)
raw_text = "OMG!!! AI students rrr brillianttt 😊"

# Simplified preprocessing used in the provided material.
preprocessed_text = "ai students are brilliant"

tokens = ["ai", "students", "are", "brilliant"]
token_ids = [101, 205, 306, 412]

print("\n" + "=" * 70)
print("1. RAW TEXT / PREPROCESSING / TOKENIZATION")
print("=" * 70)
print("Raw text       :", raw_text)
print("Preprocessed   :", preprocessed_text)
print("Tokens         :", tokens)
print("Token IDs      :", token_ids)


# ============================================================
# 2. EMBEDDING LOOKUP
# ============================================================

# d_model = 4
embedding_table = {
    101: np.array([0.20, 0.40, 0.10, 0.50]),  # ai
    205: np.array([0.60, 0.10, 0.80, 0.30]),  # students
    306: np.array([0.10, 0.70, 0.20, 0.90]),  # are
    412: np.array([0.90, 0.30, 0.60, 0.20]),  # brilliant
}

X = np.vstack([embedding_table[i] for i in token_ids])

print("\n" + "=" * 70)
print("2. EMBEDDING MATRIX X")
print("=" * 70)
print(X)


# ============================================================
# 3. POSITIONAL ENCODING
# ============================================================

d_model = 4
positions = np.arange(len(tokens))

PE = np.zeros((len(tokens), d_model))

# Standard sinusoidal positional encoding used in the PDF.
for pos in positions:
    PE[pos, 0] = math.sin(pos)
    PE[pos, 1] = math.cos(pos)
    PE[pos, 2] = math.sin(pos / 10000 ** (2 / d_model))
    PE[pos, 3] = math.cos(pos / 10000 ** (2 / d_model))

Z = X + PE

print("\n" + "=" * 70)
print("3. POSITIONAL ENCODING PE")
print("=" * 70)
print(PE)

print("\nFinal Transformer input Z = X + PE:")
print(Z)


# ============================================================
# 4. QUERY, KEY, VALUE MATRICES
# ============================================================

# Matrices shown in the provided material.
WQ = np.array([
    [1, 0],
    [0, 1],
    [1, 0],
    [0, 1]
], dtype=float)

WK = np.array([
    [1,   0],
    [0,   1],
    [0.5, 0],
    [0, 0.5]
], dtype=float)

WV = np.array([
    [0.5, 0],
    [0,   1],
    [1,   0],
    [0, 0.5]
], dtype=float)

Q = Z @ WQ
K = Z @ WK
V = Z @ WV

print("\n" + "=" * 70)
print("4. QUERY (Q), KEY (K), VALUE (V)")
print("=" * 70)
print("Q =")
print(Q)

print("\nK =")
print(K)

print("\nV =")
print(V)


# ============================================================
# 5. SCALED DOT-PRODUCT SELF-ATTENTION
# ============================================================

def softmax(x, axis=-1):
    """Numerically stable softmax."""
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


# Raw attention/relevance scores.
S = Q @ K.T

# Key/query dimension = 2.
d_k = Q.shape[1]

# Scale scores by sqrt(d_k).
S_scaled = S / math.sqrt(d_k)

# Convert scores to row-wise attention probabilities.
A = softmax(S_scaled, axis=1)

print("\n" + "=" * 70)
print("5. SELF-ATTENTION")
print("=" * 70)

print("Raw score matrix S = QK^T:")
print(S)

print("\nScaled scores S' = S / sqrt(d_k):")
print(S_scaled)

print("\nAttention matrix A = softmax(S'):")
print(A)

print("\nAttention rows sum to:")
print(A.sum(axis=1))


# ============================================================
# 6. APPLY ATTENTION WEIGHTS TO VALUES
# ============================================================

O = A @ V

print("\n" + "=" * 70)
print("6. ATTENTION OUTPUT O = AV")
print("=" * 70)
print(O)


# ============================================================
# 7. OUTPUT PROJECTION
# ============================================================

# Attention produces 2-dimensional vectors.
# Project them back to d_model = 4.
WO = np.array([
    [1, 0, 0, 1],
    [0, 1, 1, 0]
], dtype=float)

H = O @ WO

print("\n" + "=" * 70)
print("7. OUTPUT PROJECTION H = OWO")
print("=" * 70)
print(H)


# ============================================================
# 8. FIRST RESIDUAL CONNECTION
# ============================================================

R = Z + H

print("\n" + "=" * 70)
print("8. FIRST RESIDUAL CONNECTION R = Z + H")
print("=" * 70)
print(R)


# ============================================================
# 9. LAYER NORMALIZATION
# ============================================================

def layer_norm(matrix, eps=1e-8):
    """
    LayerNorm independently normalizes each token vector.
    Mean -> approximately 0
    Variance -> approximately 1
    """
    mean = np.mean(matrix, axis=1, keepdims=True)
    variance = np.var(matrix, axis=1, keepdims=True)
    return (matrix - mean) / np.sqrt(variance + eps)


Y = layer_norm(R)

print("\n" + "=" * 70)
print("9. FIRST LAYERNORM")
print("=" * 70)
print(Y)

print("\nFor AI:")
print("Mean     =", np.mean(R[0]))
print("Variance =", np.var(R[0]))
print("Y_AI     =", Y[0])


# ============================================================
# 10. FEED-FORWARD NETWORK (FFN)
# ============================================================

# d_model = 4
# d_ff = 3
W1 = np.array([
    [1,  0, 1],
    [0,  1, 1],
    [1,  1, 0],
    [1, -1, 1]
], dtype=float)

W2 = np.array([
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [1, 1, 0, 1]
], dtype=float)


def gelu_exact(x):
    """Exact GELU using the error function."""
    return 0.5 * x * (1 + np.vectorize(math.erf)(x / math.sqrt(2)))


def gelu_teaching_approx(x):
    """
    GELU approximation used for a simple classroom demonstration.

    The supplied PDF explicitly gives the AI example:
        [-1.414, 1.382, 0.016]
        -> approximately [-0.112, 1.251, 0.008]

    The standard exact GELU is also printed below for comparison.
    """
    # Common tanh approximation to GELU.
    return 0.5 * x * (
        1 + np.tanh(
            math.sqrt(2 / math.pi) * (x + 0.044715 * x**3)
        )
    )


hidden = Y @ W1
G_exact = gelu_exact(hidden)
FFN_exact = G_exact @ W2

# The PDF uses a rounded teaching value for GELU in its worked AI example.
# Use the standard GELU calculation for the actual implementation.
FFN = FFN_exact

print("\n" + "=" * 70)
print("10. FEED-FORWARD NETWORK")
print("=" * 70)

print("Hidden = YW1:")
print(hidden)

print("\nGELU(hidden) using standard GELU:")
print(G_exact)

print("\nFFN(Y) = GELU(YW1)W2:")
print(FFN)


# ============================================================
# 11. SECOND RESIDUAL CONNECTION
# ============================================================

R2 = Y + FFN

print("\n" + "=" * 70)
print("11. SECOND RESIDUAL CONNECTION R2 = Y + FFN(Y)")
print("=" * 70)
print(R2)


# ============================================================
# 12. SECOND LAYER NORMALIZATION
# ============================================================

Z_out = layer_norm(R2)

print("\n" + "=" * 70)
print("12. FINAL TRANSFORMER OUTPUT")
print("=" * 70)
print(Z_out)


# ============================================================
# 13. INTERPRETATION FOR THE TOKEN "AI"
# ============================================================

print("\n" + "=" * 70)
print("13. AI ATTENTION INTERPRETATION")
print("=" * 70)

ai_attention = A[0]

for token, weight in zip(tokens, ai_attention):
    print(f"AI -> {token:10s}: {weight * 100:6.2f}%")

print("\nThe attention mechanism transforms:")
print("Raw relevance scores -> normalized attention weights ->")
print("weighted Value information -> contextual representation.")


# ============================================================
# 14. SIMPLE Q -> K -> V DEMONSTRATION FROM PAGE 5
# ============================================================

print("\n" + "=" * 70)
print("14. SIMPLE Q -> K -> V EXAMPLE")
print("=" * 70)

Q_students = np.array([1, 2], dtype=float)

K_ai = np.array([1, 0], dtype=float)
K_students = np.array([1, 2], dtype=float)
K_brilliant = np.array([0, 2], dtype=float)

V_ai = np.array([1, 0], dtype=float)
V_students = np.array([0, 2], dtype=float)
V_brilliant = np.array([2, 1], dtype=float)

simple_scores = np.array([
    Q_students @ K_ai,
    Q_students @ K_students,
    Q_students @ K_brilliant
])

simple_attention = softmax(simple_scores)

O_students = (
    simple_attention[0] * V_ai
    + simple_attention[1] * V_students
    + simple_attention[2] * V_brilliant
)

print("Q_students =", Q_students)
print("Scores [AI, students, brilliant] =", simple_scores)
print("Attention weights =", simple_attention)
print("O_students =", O_students)


# ============================================================
# 15. COMPLETE FORMULA
# ============================================================

print("\n" + "=" * 70)
print("COMPLETE SELF-ATTENTION FORMULA")
print("=" * 70)
print("Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V")
print("\nEncoder flow:")
print("X -> X + PE -> Q,K,V -> QK^T -> scale -> softmax")
print("-> AV -> output projection -> residual -> LayerNorm")
print("-> FFN -> residual -> LayerNorm -> contextual representation")



1. RAW TEXT / PREPROCESSING / TOKENIZATION
Raw text       : OMG!!! AI students rrr brillianttt 😊
Preprocessed   : ai students are brilliant
Tokens         : ['ai', 'students', 'are', 'brilliant']
Token IDs      : [101, 205, 306, 412]

2. EMBEDDING MATRIX X
[[0.2 0.4 0.1 0.5]
 [0.6 0.1 0.8 0.3]
 [0.1 0.7 0.2 0.9]
 [0.9 0.3 0.6 0.2]]

3. POSITIONAL ENCODING PE
[[ 0.      1.      0.      1.    ]
 [ 0.8415  0.5403  0.01    1.    ]
 [ 0.9093 -0.4161  0.02    0.9998]
 [ 0.1411 -0.99    0.03    0.9996]]

Final Transformer input Z = X + PE:
[[ 0.2     1.4     0.1     1.5   ]
 [ 1.4415  0.6403  0.81    1.3   ]
 [ 1.0093  0.2839  0.22    1.8998]
 [ 1.0411 -0.69    0.63    1.1996]]

4. QUERY (Q), KEY (K), VALUE (V)
Q =
[[0.3    2.9   ]
 [2.2515 1.9403]
 [1.2293 2.1837]
 [1.6711 0.5096]]

K =
[[ 0.25    2.15  ]
 [ 1.8465  1.2903]
 [ 1.1193  1.2338]
 [ 1.3561 -0.0902]]

V =
[[ 0.2     2.15  ]
 [ 1.5307  1.2903]
 [ 0.7246  1.2338]
 [ 1.1506 -0.0902]]

5. SELF-ATTENTION
Raw score matrix S = QK^T:
[[